In [1]:
import findspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.functions import (col, when, regexp_extract, count, sum, avg, coalesce, lit, desc, expr, trim, row_number, countDistinct)
from pyspark.sql.types import (StructType, StructField, LongType, DecimalType, StringType)
from pyspark.sql.window import Window

findspark.init()
spark = SparkSession.builder.appName("Desafio_Vaga_Junior").getOrCreate()

## Criação Dataframe

In [2]:
schema_pedidos = StructType([
    StructField("id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("value", DecimalType(5,2), True),
    StructField("_corrupt_record", StringType(), True)
])

schema_clientes = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])


df_pedidos = spark.read \
    .schema(schema_pedidos) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json("test-tecnico-engenheiro-dados/data/pedidos/data.json")

df_clientes = spark.read \
    .schema(schema_clientes) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json("test-tecnico-engenheiro-dados/data/clients/data.json")

In [3]:
df_pedidos.show(5)

+-----+---------+-----+---------------+
|   id|client_id|value|_corrupt_record|
+-----+---------+-----+---------------+
| 2829|   123456|82.64|           NULL|
| 4337|   123456|16.72|           NULL|
| 5146|   123456|10.54|           NULL|
| 9784|   123456|16.25|           NULL|
|12808|   123456|30.55|           NULL|
+-----+---------+-----+---------------+
only showing top 5 rows


In [4]:
df_clientes.show(5)

+------+----------------+---------------+
|    id|            name|_corrupt_record|
+------+----------------+---------------+
|123456|   Inês Siqueira|           NULL|
|  7747|    Zachary Reis|           NULL|
|  6721| Yasmin Carvalho|           NULL|
|  3091|Sebastião Toledo|           NULL|
|  9458|   Larissa Braga|           NULL|
+------+----------------+---------------+
only showing top 5 rows


## Data Quality - Relatório de Falhas

#### Pedidos com Falhas

In [5]:
df_falhas_pedidos = (
    df_pedidos
    .withColumn(
        "id_extraido_corrupt",
        regexp_extract(col("_corrupt_record"), r'"id"\s*:\s*"?(.*?)"?[,}]', 1)
    )
    .withColumn(
        "id_relatorio",
        when(col("_corrupt_record").isNotNull(), col("id_extraido_corrupt"))
        .otherwise(col("id").cast("string"))
    )
    .withColumn(
        "motivo",
        when(col("_corrupt_record").isNotNull(), lit("json_malformado"))
        .when(col("id").isNull(), lit("id_nulo"))
        .when(col("client_id").isNull(), lit("client_id_nulo"))
        .when(col("value").isNull(), lit("value_nulo"))
        .when(col("value") < lit(0).cast(DecimalType(11, 2)), lit("value_negativo"))
    )
    .filter(col("motivo").isNotNull())
    .select(
        col("id_relatorio").alias("id"),
        col("motivo")
    )
)

df_falhas_pedidos.show(5, truncate=False)

+------+--------------+
|id    |motivo        |
+------+--------------+
|154475|value_negativo|
|182109|value_negativo|
|226706|value_negativo|
|259261|value_negativo|
|291638|value_negativo|
+------+--------------+
only showing top 5 rows


#### Pedidos Validos

In [6]:
df_pedidos_validos = (
    df_pedidos
    .filter(col("_corrupt_record").isNull())
    .filter(col("id").isNotNull())
    .filter(col("client_id").isNotNull())
    .filter(col("value").isNotNull())
    .filter(col("value") >= lit(0).cast(DecimalType(11, 2)))
    .select("id", "client_id", "value")
)

#### Pedidos Duplicados

In [7]:
df_pedidos_dedup = df_pedidos_validos.dropDuplicates(["id", "client_id", "value"])

df_resumo_ids = (
    df_pedidos_dedup
    .groupBy("id")
    .agg(
        count("*").alias("qtd_linhas"),
        countDistinct("client_id").alias("qtd_clientes_distintos"),
        countDistinct("value").alias("qtd_valores_distintos"),
        sum("value").alias("valor_total_id")
    )
)

df_ids_seguros = (
    df_resumo_ids
    .filter(
        (col("qtd_clientes_distintos") == 1) &
        (col("qtd_valores_distintos") == 1)
    )
    .select("id")
)

df_ids_inconsistentes = (
    df_resumo_ids
    .filter(
        (col("qtd_clientes_distintos") > 1) |
        (col("qtd_valores_distintos") > 1)
    )
)

df_pedidos_confiaveis = (
    df_pedidos_dedup
    .join(df_ids_seguros, on="id", how="inner")
)

df_relatorio_ids_inconsistentes = (
    df_ids_inconsistentes
    .withColumn(
        "motivo",
        when(
            (col("qtd_clientes_distintos") > 1) & (col("qtd_valores_distintos") > 1),
            lit("mesmo_id_com_clientes_e_valores_diferentes")
        )
        .when(
            col("qtd_clientes_distintos") > 1,
            lit("mesmo_id_com_clientes_diferentes")
        )
        .when(
            col("qtd_valores_distintos") > 1,
            lit("mesmo_id_com_valores_diferentes")
        )
    )
    .select(col("id"), col("motivo"))
)

df_erros = df_relatorio_ids_inconsistentes.unionByName(df_falhas_pedidos)
df_erros.show(5, truncate = False)

+--------+------------------------------------------+
|id      |motivo                                    |
+--------+------------------------------------------+
|80933114|mesmo_id_com_valores_diferentes           |
|70088478|mesmo_id_com_clientes_e_valores_diferentes|
|30861962|mesmo_id_com_clientes_e_valores_diferentes|
|245196  |mesmo_id_com_clientes_e_valores_diferentes|
|51958207|mesmo_id_com_clientes_e_valores_diferentes|
+--------+------------------------------------------+
only showing top 5 rows


#### Clientes com Falhas

In [8]:
df_falhas_clientes = (
    df_clientes
    .withColumn(
        "id_extraido_corrupt",
        regexp_extract(col("_corrupt_record"), r'"id"\s*:\s*"?(.*?)"?[,}]', 1)
    )
    .withColumn(
        "id_relatorio",
        when(col("_corrupt_record").isNotNull(), col("id_extraido_corrupt"))
        .otherwise(col("id").cast("string"))
    )
    .withColumn(
        "motivo",
        when(col("_corrupt_record").isNotNull(), lit("json_malformado"))
        .when(col("id").isNull(), lit("id_nulo"))
        .when(col("name").isNull() | (trim(col("name")) == ""), lit("name_nulo_ou_vazio"))
    )
    .filter(col("motivo").isNotNull())
    .select(
        col("id_relatorio").alias("id"),
        col("motivo")
    )
)

print("RELATÓRIO DE FALHAS - CLIENTES")
df_falhas_clientes.show(20, truncate=False)

RELATÓRIO DE FALHAS - CLIENTES
+---+------+
|id |motivo|
+---+------+
+---+------+



#### Clientes Validos

In [9]:
df_clientes_validos = (
    df_clientes
    .filter(col("_corrupt_record").isNull())
    .filter(col("id").isNotNull() & trim(col("id")).rlike(r"^\d+$"))
    .filter(col("name").isNotNull() & (trim(col("name")) != ""))
    .select(col("id"), trim(col("name")).alias("name"))
)

df_clientes_validos.show(5)

+------+----------------+
|    id|            name|
+------+----------------+
|123456|   Inês Siqueira|
|  7747|    Zachary Reis|
|  6721| Yasmin Carvalho|
|  3091|Sebastião Toledo|
|  9458|   Larissa Braga|
+------+----------------+
only showing top 5 rows


#### Clientes Duplicados

In [10]:
df_clientes_duplicados = (
    df_clientes_validos
    .groupBy("id")
    .agg(count("*").alias("qtd"))
    .filter(col("qtd") > 1)
)

print("CLIENTES DUPLICADOS")
df_clientes_validos.join(df_clientes_duplicados, on="id", how="inner").orderBy(desc("id")).show(truncate=False)

CLIENTES DUPLICADOS
+---+----+---+
|id |name|qtd|
+---+----+---+
+---+----+---+



## Agregação de Dados

In [11]:
df_agg_pedidos = (
    df_pedidos_confiaveis
    .groupBy("client_id")
    .agg(
        count("*").alias("qtd_pedidos"),
        sum("value").cast(DecimalType(11,2)).alias("valor_total_pedidos")
    )
)

df_agg_pedidos.show(5, truncate = False)

+---------+-----------+-------------------+
|client_id|qtd_pedidos|valor_total_pedidos|
+---------+-----------+-------------------+
|7747     |67         |2826.08            |
|2040     |54         |2466.23            |
|3091     |58         |3193.24            |
|9715     |50         |2531.42            |
|8484     |48         |2349.22            |
+---------+-----------+-------------------+
only showing top 5 rows


In [12]:
df_analise_clientes = (
    df_clientes_validos.alias("c")
    .join(
        df_agg_pedidos.alias("p"),
        col("c.id") == col("p.client_id"),
        "left"
    )
    .select(
        col("c.name").alias("nome_cliente"),
        coalesce(col("p.qtd_pedidos"), lit(0)).alias("quantidade_pedidos"),
        coalesce(
            col("p.valor_total_pedidos"),
            lit(0).cast(DecimalType(11, 2))
        ).alias("valor_total_pedidos")
    )
    .orderBy(desc("valor_total_pedidos"))
)



In [13]:
df_analise_clientes.show(5, truncate = False)

+-------------+------------------+-------------------+
|nome_cliente |quantidade_pedidos|valor_total_pedidos|
+-------------+------------------+-------------------+
|Inês Siqueira|494418            |24946507.01        |
|Zachary Reis |64                |4213.80            |
|Vitor Marques|72                |4185.48            |
|Inês Siqueira|65                |3977.05            |
|Sofia Castro |72                |3954.30            |
+-------------+------------------+-------------------+
only showing top 5 rows


## Analise Estatística

In [14]:
df_metricas = (
    df_analise_clientes
    .agg(
        avg(col("valor_total_pedidos")).alias("media_aritmetica"),
        expr("percentile_approx(valor_total_pedidos, 0.5, 10000)").alias("mediana"),
        expr("percentile_approx(valor_total_pedidos, 0.1, 10000)").alias("percentil_10"),
        expr("percentile_approx(valor_total_pedidos, 0.9, 10000)").alias("percentil_90")
    )
)

df_metricas = (
    df_metricas
    .withColumn("mediana_pct_media", col("mediana") / col("media_aritmetica"))
    .withColumn("p10_pct_media", col("percentil_10") / col("media_aritmetica"))
    .withColumn("p90_pct_media", col("percentil_90") / col("media_aritmetica"))
    .select(
        col("media_aritmetica").cast(DecimalType(11, 2)),
        col("mediana").cast(DecimalType(11, 2)),
        col("percentil_10").cast(DecimalType(11, 2)),
        col("percentil_90").cast(DecimalType(11, 2)),
        round(col("mediana_pct_media") * 100, 2).alias("mediana_%"),
        round(col("p10_pct_media") * 100, 2).alias("p10_%"),
        round(col("p90_pct_media") * 100, 2).alias("p90_%")
    )
)

print("MÉTRICAS COM PERCENTUAL")
df_metricas.select("media_aritmetica","mediana","mediana_%","percentil_10","p10_%","percentil_90","p90_%").show(truncate=False)

MÉTRICAS COM PERCENTUAL
+----------------+-------+---------+------------+-----+------------+-----+
|media_aritmetica|mediana|mediana_%|percentil_10|p10_%|percentil_90|p90_%|
+----------------+-------+---------+------------+-----+------------+-----+
|4996.23         |2491.19|49.86    |1989.83     |39.83|3025.98     |60.57|
+----------------+-------+---------+------------+-----+------------+-----+



## Filtragem - Acima da Média

In [15]:
df_acima_media = (
    df_analise_clientes.crossJoin(df_metricas)
    .filter(col("valor_total_pedidos") > col("media_aritmetica"))
    .select(
        "nome_cliente",
        "quantidade_pedidos",
        "valor_total_pedidos"
    )
    .orderBy(desc("valor_total_pedidos"))
)

print("CLIENTES ACIMA DA MÉDIA")
df_acima_media.show(5, truncate=False)

CLIENTES ACIMA DA MÉDIA
+-------------+------------------+-------------------+
|nome_cliente |quantidade_pedidos|valor_total_pedidos|
+-------------+------------------+-------------------+
|Inês Siqueira|494418            |24946507.01        |
+-------------+------------------+-------------------+



## Filtragem - Média Truncada

In [17]:
df_media_truncada = (
    df_analise_clientes.crossJoin(df_metricas)
    .filter(col("valor_total_pedidos") >= col("percentil_10"))
    .filter(col("valor_total_pedidos") <= col("percentil_90"))
    .select(
        "nome_cliente",
        "quantidade_pedidos",
        "valor_total_pedidos"
    )
    .orderBy(desc("valor_total_pedidos"))
)


print("CLIENTES ENTRE P10 E P90")
df_media_truncada.show(10, truncate=False)

CLIENTES ENTRE P10 E P90
+--------------+------------------+-------------------+
|nome_cliente  |quantidade_pedidos|valor_total_pedidos|
+--------------+------------------+-------------------+
|Zachary Reis  |60                |3025.98            |
|Tereza Leal   |60                |3025.93            |
|Julio Viana   |63                |3025.73            |
|Rafaela Vieira|55                |3025.62            |
|Heloísa Brito |64                |3025.41            |
|Inês Siqueira |53                |3025.29            |
|Marcos Dias   |52                |3024.83            |
|Elaine Coelho |56                |3024.20            |
|Caio Teles    |52                |3023.65            |
|Sérgio Esteves|55                |3023.06            |
+--------------+------------------+-------------------+
only showing top 10 rows
